# Gasoline (RBOB) Flat Price Contracts — Open Interest EDA

Analyze the relative size of each Gasoline flat price contract code from `disaggregated_combined.csv`.

Flat price codes (from `docs/gasoline_cot_mapping.md`):
- `111659` — GASOLINE BLENDSTOCK (RBOB) - NEW YORK MERCANTILE EXCHANGE (benchmark)
- `11165J` — RBOB GASOLINE FINANCIAL - NEW YORK MERCANTILE EXCHANGE
- `111416` — RBOB GASOLINE 1ST LINE - ICE FUTURES ENERGY DIV

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [2]:
df = pd.read_csv("../../downloads/cftc/disaggregated_combined.csv", low_memory=False)

XB_FLAT_PRICE_CODES = [
    "111659",  # GASOLINE BLENDSTOCK (RBOB) - NYMEX (benchmark)
    "11165J",  # RBOB GASOLINE FINANCIAL - NYMEX
    "111416",  # RBOB GASOLINE 1ST LINE - ICE FUTURES ENERGY DIV
]

xb = df[df["cftc_contract_market_code"].isin(XB_FLAT_PRICE_CODES)].copy()
xb["report_date_as_yyyy_mm_dd"] = pd.to_datetime(xb["report_date_as_yyyy_mm_dd"])
xb["open_interest_all"] = pd.to_numeric(xb["open_interest_all"], errors="coerce")
print(f"Gasoline flat price rows: {len(xb):,}")
print(f"Date range: {xb['report_date_as_yyyy_mm_dd'].min().date()} to {xb['report_date_as_yyyy_mm_dd'].max().date()}")
print(f"Codes found: {sorted(xb['cftc_contract_market_code'].unique())}")

Gasoline flat price rows: 1,055
Date range: 2006-06-13 to 2026-03-24
Codes found: ['111416', '111659', '11165J']


## Average Open Interest by Contract Code

In [4]:
code_labels = (
    xb.groupby("cftc_contract_market_code")["market_and_exchange_names"]
    .first()
    .to_dict()
)

avg_oi = (
    xb.groupby("cftc_contract_market_code")["open_interest_all"]
    .mean()
    .sort_values(ascending=False)
    .to_frame("avg_open_interest")
)
avg_oi["contract_name"] = avg_oi.index.map(code_labels)
avg_oi["source"] = "CFTC"
avg_oi["pct_of_total"] = (avg_oi["avg_open_interest"] / avg_oi["avg_open_interest"].sum() * 100)
avg_oi["avg_open_interest"] = avg_oi["avg_open_interest"].round(0).astype(int)
avg_oi["pct_of_total"] = avg_oi["pct_of_total"].round(2)

avg_oi[["contract_name", "source", "avg_open_interest", "pct_of_total"]]

,contract_name,source,avg_open_interest,pct_of_total
cftc_contract_market_code,,,,
111659,GASOLINE BLENDSTOCK (RBOB) - NEW YORK MERCANTI...,CFTC,333240,95.21
111416,RBOB GASOLINE 1ST LINE - ICE FUTURES ENERGY DIV,CFTC,9177,2.62
11165J,RBOB GASOLINE FINANCIAL - NEW YORK MERCANTILE ...,CFTC,7591,2.17


In [5]:
fig = px.pie(
    avg_oi.reset_index(),
    values="avg_open_interest",
    names="cftc_contract_market_code",
    hover_data=["contract_name"],
    title="Gasoline (RBOB) Flat Price Contracts — Average Open Interest Share",
)
fig.update_traces(textinfo="label+percent", textposition="outside")
fig.show()

## Open Interest Over Time by Contract Code

In [6]:
fig = px.line(
    xb.sort_values("report_date_as_yyyy_mm_dd"),
    x="report_date_as_yyyy_mm_dd",
    y="open_interest_all",
    color="cftc_contract_market_code",
    hover_data=["market_and_exchange_names"],
    title="Gasoline (RBOB) Flat Price Contracts — Open Interest Over Time",
    labels={
        "report_date_as_yyyy_mm_dd": "Report Date",
        "open_interest_all": "Open Interest",
        "cftc_contract_market_code": "Contract Code",
    },
)
fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.3))
fig.show()

## Open Interest Over Time — Stacked Area (Percentage)

In [7]:
pivot = xb.pivot_table(
    index="report_date_as_yyyy_mm_dd",
    columns="cftc_contract_market_code",
    values="open_interest_all",
    aggfunc="sum",
).fillna(0)

pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

fig = go.Figure()
for col in pct.columns:
    label = code_labels.get(col, col)
    fig.add_trace(go.Scatter(
        x=pct.index, y=pct[col],
        name=col,
        hovertext=label,
        stackgroup="one",
        groupnorm="percent",
    ))
fig.update_layout(
    title="Gasoline (RBOB) Flat Price Contracts — OI Share Over Time (%)",
    xaxis_title="Report Date",
    yaxis_title="% of Total Gasoline Flat Price OI",
    legend=dict(orientation="h", yanchor="bottom", y=-0.3),
)
fig.show()

## Summary Table — Latest Reporting Date

In [8]:
latest_date = xb["report_date_as_yyyy_mm_dd"].max()
latest = xb[xb["report_date_as_yyyy_mm_dd"] == latest_date].copy()

summary = (
    latest.groupby("cftc_contract_market_code")
    .agg(
        contract_name=("market_and_exchange_names", "first"),
        open_interest=("open_interest_all", "sum"),
    )
    .sort_values("open_interest", ascending=False)
)
summary["pct_of_total"] = (summary["open_interest"] / summary["open_interest"].sum() * 100).round(2)

print(f"Latest report date: {latest_date.date()}")
summary

Latest report date: 2026-03-24


,contract_name,open_interest,pct_of_total
cftc_contract_market_code,,,
111659,GASOLINE RBOB - NEW YORK MERCANTILE EXCHANGE,357477,100.0
